## AI4Climate ML tutorial - template
* Author: <INSERT AUTHOR>
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-02-10
* © British Crown Copyright 2017-2065, Met Office. Please see LICENSE.md for license details.

## Overview of the broad topic covered

In this notebook, we will use the previously prepared tabular dataset to predict climate zones. We will train on different eras to see how well the results generalise with climate change. 

### Prerequisites 
what background information is needed to go through the notebook
- Same as previouis notebooks
- Have completed data exploration notebook

### Learning outcomes from completing the notebook

- Understand the key components of a machine learning training pipeline and how they fit together
- Understand key terminology in describing the machine learning pipeline
- Initial understanding of how to choose appropriate components for each stage in the pipeline.

## Tutorial - Key Elements of a Machine Learning Pipeline

Using a series of data science and machine learning and algorithms to go from input data to a series of predictions is usually referred to as a pipeline. In this noteboook we will be exploring the key components of such a pipeline in constructing and training a machine learning algorithm with some input data.

In this notebook we will look at 2 pipeline, one for a supervised classification problem, and the other for an unsupervised clustering problem.

The steps we willl go through are as follows:
1. Data Loading & Cleaning - Start by loading the data, and filtering out any data considered to be unsuitable training and evaluation of machine learning algorithms. Selection of appropriate data is an important way in which domain expertise in vital in getting good results.
2. Feature Engineering - The first step is to prepare the data for presenting to the algorithm. Different ways of presenting the data will emphasise different features, and choosing the right features is important for getting good results. Knowledge of what features represent based on domain knowledge is again very important.
3. Train/test Split - Before we train the algorithm, we need to split into train and test sets. This is to ensure out algorithm doesn't overfit, learning irrelevant details that are not representative of the whole space of possible data, but rather that in generalises well.
4. Data Preparation - The machine learning algorithm only sees numbers as numbers, with no inherent understanding of meaning or context. We need to ensure different features are scaled to be comparable, otherwise big numbers will be treated as more important by the algorithm, irrespective of what those numbers mean. Value are typically scaled to a range of [0,1] or, assuming a gaussian distribution, to have mean=0 and std_dev=1.
5. Algorithm Setup - Here we select the particular algorithm e,.g. neural network, k-means clustering, and specify the hyperparameters. It is important to distinguish between parameters and hyperparameters.
    - Parameters are the values that calculated by the training process.
    - Hyperparameters are values specified in algorithm setup, which are not altered by training. These need to be fine-tuned using an additional outer training loop called hyperparameter tuning.
6. Algorithm Training - Execute the algorithm to calculate the best parameters for the chosen ML algorithm to fit the supplied training data
7. Inference - Once we have an algorithm, we use it to produce predictions, for both the train and test sets.
8. Evaluation - We then compare the predictions of the trained algorithms to expected results. For supervised learning, this will be supplied target values. For unsupervised learning, we will explore the results and their usefulness much like in exploratory data analysis.
9. Interpretability & Explainability - Machine learning models are often treated as black-boxes, that is we can't know or understand how or why the algoriothm produces a particular output. Explainability and interpretability aim to change this, giving insight on the internals of the algorithms (explainability) and guidance on interpeting a particular result (interpretability).
10. Model Storage - Model training can be an expensive process that we don't want to perform too often, and. once we have a model that performs well we save its state so it can be reloaded and used subsequently for inference on later problems.


### Key Terms

- *supervised learning* - training an algorithm to map from input to target data or labels.
- *unsupervised learning* - training an algorithm to find structure in data that has no labels.
- *regression* - An algorithm that predicts a continuous values.
- *classification* - An algorithm that predicts from a set of discrete values
- *metric* - A measure of the performance of the ML algorithm.
- *parameter* - A value in the algorithm that is determined by the training process e.g. neural network weights or decision tree thresholds.
- *hyperparameter* - A value in the algorithm that is not determined by training and must be specified or tuned. e.g. number of hidden layers or max number of decision tree levels.
- *feature engineering* - the process of creating input variables for the ML aglorithm that will give desirable results.
- *training set* - The subset of your data that you use for training your algorithm.
- *validation set* - The subset of your data that you use for testing your trained algorithm and which informs subsequent development to improve results.
- *test set*- The subset of your data that you set aside and do not use while developing the algorithm. Once the development process is finished, you check the final result with
this subset of the data to check it truly generalises to unseen data.
*inference* - Calculating predictions from input data used a trained algorithm.

More information on jargon
- [Google Machine Learning Glossary](https://developers.google.com/machine-learning/glossary)
- [ML Cheatsheet](https://ml-cheatsheet.readthedocs.io/en/latest/glossary.html)


## Setup
First we start by loading the data we have prepared previously, and other set up elements

### Imports

In [1]:
import pathlib
import os
import datetime
import json

In [2]:
import pandas

In [3]:
import mlflow

In [4]:
mlflow.sklearn.autolog()

In [5]:
import sklearn
import sklearn.preprocessing
import sklearn.tree
import torch

## Step 1 - Data Loading and Cleaning

#### Dataset parameters
Start by defininng the parameters to use for loading the dataset. These have been collected into a config file for use across the notebooks in this tutorial material.

In [6]:
with open ('config.json','r') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

{'platform': 'mo_linux',
 'default_dirs': {'mo_linux': '/data/users/dscop/ml_tutorial/',
  'jasmin': '/gws/nopw/j04/mohc_shared/dscop/'},
 'climate_subgroups': {'non-land': 'None',
  'Af': 'Tropical, rainforest',
  'Am': 'Tropical, monsoon',
  'Aw': 'Tropical, savannah',
  'BWh': 'Arid, desert, hot',
  'BWk': 'Arid, desert, cold',
  'BSh': 'Arid, steppe, hot',
  'BSk': 'Arid, steppe, cold',
  'Csa': 'Temperate, dry summer, hot summer',
  'Csb': 'Temperate, dry summer, warm summer',
  'Csc': 'Temperate, dry summer, cold summer',
  'Cwa': 'Temperate, dry winter, hot summer',
  'Cwb': 'Temperate, dry winter, warm summer',
  'Cwc': 'Temperate, dry winter, cold summer',
  'Cfa': 'Temperate, no dry season, hot summer',
  'Cfb': 'Temperate, no dry season, warm summer',
  'Cfc': 'Temperate, no dry season, cold summer',
  'Dsa': 'Cold, dry summer, hot summer',
  'Dsb': 'Cold, dry summer, warm summer',
  'Dsc': 'Cold, dry summer, cold summer',
  'Dsd': 'Cold, dry summer, very cold winter',
  'Dw

In [7]:
def get_platform_dir(select_platform, config):
    try:
        root_path = pathlib.Path(config['default_dirs'][select_platform]) / 'climate_zones'
    except KeyError:
        root_path = pathlib.Path(os.environ['HOME']) / 'climate_zones'
    return root_path

In [8]:
current_platform = tutorial_config['platform']

In [9]:
root_data_dir = get_platform_dir(current_platform, tutorial_config)

print(root_data_dir.is_dir())
root_data_dir

True


PosixPath('/data/users/dscop/ml_tutorial/climate_zones')

In [10]:
ml_ready_dir = root_data_dir / 'ml_ready'
print(ml_ready_dir.is_dir())
ml_ready_dir

True


PosixPath('/data/users/dscop/ml_tutorial/climate_zones/ml_ready')

In [11]:
resolutions_dict = {float(k1): v1 for k1,v1 in tutorial_config['resolutions_names'].items()}

dataset_prefix_dict = tutorial_config['dataset_prefix']

format_str = 'nc'
historic_scenario_str = 'historic'

future_scenario_list = tutorial_config['future_scenarios']
historic_scenario_list = tutorial_config['historic_scenarios']

time_periods = { 
    (1901,1930): historic_scenario_list, 
    (1931,1960): historic_scenario_list,
    (1961,1990): historic_scenario_list,
    (1991,2020): historic_scenario_list,
    (2041,2070): future_scenario_list,
    (2071,2099): future_scenario_list,
}


In [12]:
fname_template = tutorial_config['fname_template']
time_dir_template = tutorial_config['time_dir_template']
ml_ready_fname_template = tutorial_config['csv_out_template']

### Load data for training

In [13]:
current_res = 1.0

In [14]:
mlready_data_path = ml_ready_dir / ml_ready_fname_template.format(resolution=resolutions_dict[current_res])
print(mlready_data_path.is_file())
mlready_data_path

True


PosixPath('/data/users/dscop/ml_tutorial/climate_zones/ml_ready/climate_zones_1p0.csv')

In [15]:
zones_df = pandas.read_csv(mlready_data_path)

In [16]:
zones_df.columns

Index(['lat', 'lon', 'precipitation_1.0_mean', 'precipitation_2.0_mean',
       'precipitation_3.0_mean', 'precipitation_4.0_mean',
       'precipitation_5.0_mean', 'precipitation_6.0_mean',
       'precipitation_7.0_mean', 'precipitation_8.0_mean',
       'precipitation_9.0_mean', 'precipitation_10.0_mean',
       'precipitation_11.0_mean', 'precipitation_12.0_mean',
       'air_temperature_1.0_mean', 'air_temperature_2.0_mean',
       'air_temperature_3.0_mean', 'air_temperature_4.0_mean',
       'air_temperature_5.0_mean', 'air_temperature_6.0_mean',
       'air_temperature_7.0_mean', 'air_temperature_8.0_mean',
       'air_temperature_9.0_mean', 'air_temperature_10.0_mean',
       'air_temperature_11.0_mean', 'air_temperature_12.0_mean',
       'precipitation_1.0_std', 'precipitation_2.0_std',
       'precipitation_3.0_std', 'precipitation_4.0_std',
       'precipitation_5.0_std', 'precipitation_6.0_std',
       'precipitation_7.0_std', 'precipitation_8.0_std',
       'precipitatio

## 2. Feature Engineering

Feature engineering is about preparing the dataset for use training learning algorithm. There are a variety of different tasks that form part of this
* Selecting a subset of data e.g. by time, spatial extent, certain variables/fields
* Transforming the data in some way e.g. changing resolution, transforming from gridded to tabular.
* Calcualting dervived variables from combinations of variables e.g. wind speed from u and v wind.
* Calcuating summary variables e.g. daily maxima or monthly means from hourly data.

### ML Ready data

This data preparation step can be very time consuming for a variety of reasons:
* Although you may know of a dataset, yo many not immediately know how to find or access it
* you often need to load and explore the data to decide what to use, especially if there is inadequate documentation
* the data may not easily work with tools that you are already familair
* you need to understand the data contents to understand how to use it in a sicnetifically valid way with machine learning tools. It is expcially important to under the limitations of the data and its usage.

The difficulty in using data with data driven techniques such as machine learning can limit the value of the data. As a result increasingly dataset creators, maintainers and curators are looking to ensure they package the data in way that is *ML-ready* or *AI-ready*. More information on what this means can be found in the repository, which includes documentation and exaples of how to assess whether your dataset in ML ready.
* [AI Data Readiness Github Repository](https://github.com/MetOffice/ai_data_readiness)

#### Climate Zones Feature Engineering

Most of the feature engineering for this dataset is done in the Data Prep notebook, so look at that notebook to see the code used. The the feature engineering process was as follows:
* We started with global gridded data fields, for a variety of historical and future periods spanning 200 years at different resolution, representing:
  * Monthly means of mean and standard deviation for air temperature and precipitation
  * Climate Zone Classificaton for each land point on earth
* We grouped our dataset by period and scenario
* We transformed from a gridded format to a tabular format. The reason for doing that is so we consider each point on the map as separate data point, rather than a whole global map. This means we have many more training examples.
* We excluded all the non-land points, as they have no climate classification
* We changed the climate zone feature from just being a number to have the proper zone description string e.g. `Af`, so that the data is more self describing.
* We calculated a second zone field, which contains only the major groupings of zones, such as `A` and `B`, rather than subgroups like `Af` or `Bsh`. This means there are only 5 classification classes which makes training the ML easier, and requires a smaller training dataset for good results.
* We did this processing for subset (time period, scenario) and added the start and end of the period and scenario name as features, so we could then concatenate all the data into a single tabular dataset.
* We then saved out the data in a relevant format.


We have the following predictors avaialable for training. We'll start by using only the climate means as predictors, and only aiming for the 5 classes, rather than the full 30. 

In [17]:
predictors_dict = {
    'precip_mean': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'mean' in c1],
    'precip_std': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'std' in c1],
    'temp_mean': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'mean' in c1],
    'temp_std': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'std' in c1],
}


In [18]:
predictors = predictors_dict['precip_mean'] + predictors_dict['temp_mean']
predictors

['precipitation_1.0_mean',
 'precipitation_2.0_mean',
 'precipitation_3.0_mean',
 'precipitation_4.0_mean',
 'precipitation_5.0_mean',
 'precipitation_6.0_mean',
 'precipitation_7.0_mean',
 'precipitation_8.0_mean',
 'precipitation_9.0_mean',
 'precipitation_10.0_mean',
 'precipitation_11.0_mean',
 'precipitation_12.0_mean',
 'air_temperature_1.0_mean',
 'air_temperature_2.0_mean',
 'air_temperature_3.0_mean',
 'air_temperature_4.0_mean',
 'air_temperature_5.0_mean',
 'air_temperature_6.0_mean',
 'air_temperature_7.0_mean',
 'air_temperature_8.0_mean',
 'air_temperature_9.0_mean',
 'air_temperature_10.0_mean',
 'air_temperature_11.0_mean',
 'air_temperature_12.0_mean']

We have two options for the target. We have the full 30 class climate subgroups of the Koppen-Geiger classification data from the original dataset. We also have the processed five class climate group target, which presents an easier target for our classification algorithm to predict.

In [19]:
target_var = 'climate_group' # 5 classes
# target_var = 'climate_subgroup # 30 classes


## 3. Train/test split

The next step is to [split the data into different subsets](https://www.geeksforgeeks.org/machine-learning/how-to-do-train-test-split-using-sklearn-in-python/) called *train*, *validation* and *test* sets. This to check how well what the algorithm learn generalises to unseen data, so it learns the true meaningful data relationship of interest, rather than suprious connections or noise present in the particular training examples. The different subsets are used as follows:
- *train* set This is used in the training loop for calculating the error in predictions with current parameters and then upating the weight towards a better prediction. For neural networks, this is done through one or other variant on the "gradient descent with back propogation" algorithm.
- *validation* or *dev* set - This is used for checking each model trained e.g. varying hyparameters like number of neurons or layer, or learning rate. The data is not seen during the triaing loop, so shows whether the model has learnt general relationships.
- *test* set - This data is put aside during model development until development is finished, and then the final model is tested using this data. The reason for this additional test is that during dev or validation process, the hyperparameters could have been selected that were particular to the data in validation set but are not representative of general performance. The final test on the test shows whether performance o the validation set represents genuine learning by the model.

 ### Considerations
 - consistency of distributions
- class imbalance
- correlation between samples

### Methods

There are various ways of selecting subsets
* Random - Specify which fraction of the dataset goes into each set and select at random
  * A typical fraction is 80%/10%10% for train/val/test.
  * One issue with this is points that are correlated may be in different sets, rsulting in *information leakage* which reduces the meaningfulness of evluation with validation and test sets. Correlation could be due to temporal or spatial proximty, or other reasons.
* Weighted - Rather than selecting at random from the data, one can divide the data by class or for particular attribute, and then randomly assign to trai, validation or test for each point. This ensure a balanced reperesention of data acrosss different sets. 
* Rebalanced - Similar to weighted, but instead of using all available data, the data is u=subsampled to provide equal memebers of each target class in the training dataset.
* Temporal - Select different time periods, for example by year.


In [ ]:
random_seed = tutorial_config['random_seed']

In [20]:
test_frac = 0.2
val_frac = 0.2

In [21]:
zones_df['test'] = False
test_df = zones_df.groupby(['period_start','scenario']).sample(frac=test_frac, random_state=random_seed)
zones_df['test'][test_df.index] = True

/var/tmp/ipykernel_321047/1662537359.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  zones_df['test'][test_df.index] = True


In [22]:
zones_df['val'] = False
val_df = zones_df[zones_df['test'] == False].groupby(['period_start','scenario']).sample(frac=(0.2)/(1.0-test_frac), random_state=random_seed)
zones_df['val'][val_df.index] = True

/var/tmp/ipykernel_321047/2859563780.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  zones_df['val'][val_df.index] = True


In [23]:
train_df = zones_df[((zones_df['val'] == False) & (zones_df['test'] == False ))]

## 4. Data Preparation
Now we prepare the data for ingestion by the ML training process. This is a about the mechanics of ML algorithms, where values for different features need to be normalised so are they are considered equally important by the algorithm.

In [24]:
train_df[predictors].describe()

,precipitation_1.0_mean,precipitation_2.0_mean,precipitation_3.0_mean,precipitation_4.0_mean,precipitation_5.0_mean,precipitation_6.0_mean,precipitation_7.0_mean,precipitation_8.0_mean,precipitation_9.0_mean,precipitation_10.0_mean,...,air_temperature_3.0_mean,air_temperature_4.0_mean,air_temperature_5.0_mean,air_temperature_6.0_mean,air_temperature_7.0_mean,air_temperature_8.0_mean,air_temperature_9.0_mean,air_temperature_10.0_mean,air_temperature_11.0_mean,air_temperature_12.0_mean
count,1.381411e+07,1.381411e+07,1.381411e+07,1.381411e+07,1.381411e+07,1.381411e+07,1.381411e+07,1.381411e+07,1.381411e+07,1.381411e+07,...,1.381411e+07,1.381411e+07,1.381411e+07,1.381411e+07,1.381411e+07,1.381411e+07,1.381411e+07,1.381411e+07,1.381411e+07,1.381411e+07
mean,4.378443e+01,4.090583e+01,4.586747e+01,4.457352e+01,4.853304e+01,5.363612e+01,6.235919e+01,6.261932e+01,5.320702e+01,4.812687e+01,...,-6.703962e+00,-4.266866e+00,-1.212646e+00,1.568796e+00,2.396590e+00,1.641435e+00,-3.323668e-01,-2.310610e+00,-3.799535e+00,-4.682084e+00
std,7.231755e+01,6.608446e+01,6.986829e+01,6.448619e+01,6.723700e+01,7.655781e+01,8.632384e+01,8.092138e+01,6.885764e+01,6.344102e+01,...,2.545930e+01,2.721423e+01,2.817385e+01,2.896717e+01,3.011571e+01,3.002818e+01,2.866975e+01,2.543900e+01,2.163863e+01,2.006238e+01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,-5.600000e+01,-6.187500e+01,-6.312500e+01,-6.243750e+01,-6.412500e+01,-6.412500e+01,-6.312500e+01,-5.531250e+01,-4.193750e+01,-4.418750e+01
25%,4.312500e+00,4.937500e+00,7.125000e+00,8.750000e+00,9.437500e+00,6.187500e+00,7.250000e+00,1.043750e+01,7.187500e+00,7.750000e+00,...,-2.643750e+01,-2.531250e+01,-2.350000e+01,-2.431250e+01,-2.581250e+01,-2.631250e+01,-2.456250e+01,-2.300000e+01,-2.231250e+01,-2.237500e+01
50%,1.625000e+01,1.575000e+01,1.975000e+01,2.175000e+01,2.587500e+01,2.768750e+01,3.587500e+01,3.900000e+01,3.262500e+01,2.693750e+01,...,-6.312500e+00,1.562500e+00,8.125000e+00,1.350000e+01,1.587500e+01,1.431250e+01,9.312500e+00,2.562500e+00,-5.562500e+00,-9.625000e+00
75%,4.393750e+01,4.000000e+01,4.781250e+01,4.737500e+01,5.637500e+01,7.068750e+01,8.156250e+01,8.025000e+01,6.712500e+01,6.093750e+01,...,1.906250e+01,2.018750e+01,2.100000e+01,2.300000e+01,2.406250e+01,2.393750e+01,2.268750e+01,2.100000e+01,1.856250e+01,1.518750e+01
max,9.738750e+02,7.685625e+02,7.616875e+02,8.018125e+02,1.069000e+03,1.864438e+03,2.010688e+03,1.487250e+03,1.010062e+03,9.937500e+02,...,3.506250e+01,3.756250e+01,3.950000e+01,4.156250e+01,4.268750e+01,4.206250e+01,3.887500e+01,3.581250e+01,3.537500e+01,3.531250e+01


We see a very different range of values. The ML algorithm doesn't "understand" the different ranges of values, we need to ensure all values are in a similar range so they can all contribute to a prediction. Ultimately it is the value relative to other values thats important, rather than the absolute value (from an ML perspective) so we can normalise the distribution to being between 0 and 1, or have a 0 mean and standard deviation of 1, which we will do here.m

In [25]:
input_scaler = sklearn.preprocessing.StandardScaler()
input_scaler.fit(train_df[predictors])


,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [26]:
input_scaler.mean_

array([43.78442941, 40.90583104, 45.86747426, 44.57351714, 48.53303549,
       53.63612305, 62.35918962, 62.61932474, 53.20701715, 48.12687422,
       43.30666625, 43.71645612, -5.73554134, -6.94449257, -6.70396209,
       -4.26686644, -1.21264619,  1.56879608,  2.39658982,  1.64143475,
       -0.3323668 , -2.31060962, -3.79953536, -4.68208377])

In [27]:
X_train = input_scaler.transform(train_df[predictors])
X_val = input_scaler.transform(val_df[predictors])
X_test = input_scaler.transform(test_df[predictors])

In [28]:
target_encoder = sklearn.preprocessing.OneHotEncoder()
target_encoder.fit(train_df[[target_var]])

,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a :class:`scipy.sparse.csr_matrix`,i.e. a sparse matrix in ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",True
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide `.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'error'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide `.",None
,"max_catego

In [29]:
train_df[[target_var]].value_counts()

climate_group
E                4796992
D                3616005
B                2617667
A                1635919
C                1147528
Name: count, dtype: int64

In [30]:
y_train = target_encoder.transform(train_df[[target_var]]).toarray()
y_val = target_encoder.transform(val_df[[target_var]]).toarray()
y_test = target_encoder.transform(test_df[[target_var]]).toarray()


## 5. Algorithm Setup

Now that we've done the "data wrangling", we can finally move on to the machine learning specific parts of the project (recieved wisdom is that 80% of a ML poroject will be spent on data wrangling!). The first step is to set up objects representing the algorithms we will be training.

For this introductory example, where the emphasis is on the concepts, rather than the final trained algorothm, we are going to use scikit-learn which abstracts away some of the details so it is easier to see how the concepts translate into code.

Another advantage is that all the different algorithms that are available through scikit-learn have a consistent interface, so we can use the same code to trian several different classifiers and compare their performance.

In this section we will specify which classifier we will use and the hyper-parameters for the different algorithms that will be trained. The actual training happens subsequently.

We will use two decision tree type algorithms, and then two neural networks of different sizes. We will look in more detail at how these two sorts of algorithms work in a subsequent section of the tutorial.

In [31]:
experiment_name = 'ai4c_climate_zone'

In [32]:
classifiers_params = {
    'decision_tree': {'class': sklearn.tree.DecisionTreeClassifier, 'opts': {'max_depth':10, 'class_weight':'balanced'}},
    'random_forest': {'class': sklearn.ensemble.RandomForestClassifier, 'opts': {'max_depth':10, 'class_weight':'balanced', 'n_estimators': 10}},
     'ann_5_100': {'class': sklearn.neural_network.MLPClassifier, 'opts': {'hidden_layer_sizes':(100,100,100,100,100)}},
     'ann_3_200': {'class': sklearn.neural_network.MLPClassifier, 'opts': {'hidden_layer_sizes':(200,200,200)}},   
}

## 7. Algorithm Training

Now that everthing is set up, we can train the chosen algorithms with the data we have prepared. Through the scikit-learn interface, one can call fit() on the algorithm object, passing the input and target data as the arguments to the function.

Typically when we are developing a solution to a problem using machine learning, we may training many different models to understand appropriate choices of dataset, architecture, hyperparameters and other elements of a proposed solution. When comparing the results for different options, it is very important to keep track of which set of choices correspond to which set of results. A common toll is an experiment tracking framework, such a ML Flow which we will use here. This can automatically record the choices in an experiment, but also the trained model, and any other artifacts you choose to record, such as plots of result, scores of evaluation metrics and anything else you wish to store as part of the experiment.

Further reading:
* [ML FLow docs](https://mlflow.org/)

In [33]:
try:
    mlflow_port = os.environ['MLFLOW_PORT']
except KeyError:
    mlflow_port = 4455    
mlflow_server_uri = f'http://localhost:{mlflow_port}'


In [34]:
print(f'connecting to mlflow server {mlflow_server_uri}')
mlflow.set_tracking_uri(mlflow_server_uri)



connecting to mlflow server http://localhost:4455


In [35]:
if mlflow.get_experiment_by_name(experiment_name) is None:
    exp_id = mlflow.create_experiment(experiment_name)
exp1 = mlflow.get_experiment_by_name(experiment_name)
exp1



<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1772200876372, experiment_id='4', last_update_time=1772200876372, lifecycle_stage='active', name='ai4c_climate_zone', tags={}>

In [ ]:
classifiers_dict = {}             
for clf_name, clf_params in classifiers_params.items():
    with mlflow.start_run(experiment_id=exp1.experiment_id) as current_run:
        clf1 = clf_params['class'](**clf_params['opts'])
        clf1.fit(X_train, y_train)
        classifiers_dict[clf_name] = clf1

2026/02/27 14:07:14 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/data/users/stephen.haddad/conda/ai4c_hack/lib/python3.14/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format."
2026/02/27 14:08:19 WARNING mlflow.sklearn.utils: log_loss failed. The metric training_log_loss will not be recorded. Metric error: Found array with dim 3, while dim <= 2 is required.
2026/02/27 14:08:20 WARNING mlflow.sklearn.utils: roc_auc_score failed. The metric training_roc_auc will not be recorded. Metric error: Found array with dim 3, while dim <= 2 is required.
2026/02/27 14:08:21 WARNING mlflow.sklearn.utils: Failed to autolog artifacts for DecisionTreeClassifier. Logging error: cannot use 'n

🏃 View run unleashed-panda-330 at: http://localhost:4455/#/experiments/4/runs/8e8629d698704f909cdc86c45caf341b
🧪 View experiment at: http://localhost:4455/#/experiments/4


2026/02/27 14:16:09 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/data/users/stephen.haddad/conda/ai4c_hack/lib/python3.14/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format."
2026/02/27 14:17:08 WARNING mlflow.sklearn.utils: log_loss failed. The metric training_log_loss will not be recorded. Metric error: Found array with dim 3, while dim <= 2 is required.
2026/02/27 14:17:09 WARNING mlflow.sklearn.utils: roc_auc_score failed. The metric training_roc_auc will not be recorded. Metric error: Found array with dim 3, while dim <= 2 is required.
2026/02/27 14:17:09 WARNING mlflow.sklearn.utils: Failed to autolog artifacts for RandomForestClassifier. Logging error: cannot use 'n

🏃 View run redolent-sponge-847 at: http://localhost:4455/#/experiments/4/runs/f718009f4a0442a8aab1048d5d725890
🧪 View experiment at: http://localhost:4455/#/experiments/4


In [ ]:
y_pred_train = {}
y_pred_val = {}
for clf_name, clf_obj in classifiers_dict.items():
    y_pred_train[clf_name] = clf_obj.predict(X_train)
    y_pred_val[clf_name]= clf_obj.predict(X_val)

## 9. Evaluation
Having trained a model, we want to evaluate how well it predicts the target values. Doing this for the training set is a god starting point for understanding how well the relationships present in the training data have been learnt. To see how well this mapping represents the general relationships of interests, rather than being specific to the sample of data present in the trianing set, the more important result is the metric scores for the validation set.

In [ ]:
for clf_name, clf_obj in classifiers_dict.items():
    sklearn.metrics.precision_recall_fscore_support(y_train, y_pred_train[clf_name])

In [ ]:
for clf_name, clf_obj in classifiers_dict.items():
    sklearn.metrics.precision_recall_fscore_support(y_train, y_pred_val[clf_name])

We will go into more details on evaluating the performance of a machine learning model in a separate notebook.

## 10. Interpretability and Explainability (also called XAI)
An increasingly important element of understand the performance of a trained model is to understand how what has been learnt has been learnt, usaully by techniques that allow us to look inside the "black box" of a machine learning algorithm.
* Interpretability involves interrogating the internal details of an algorithm to understand why a prediction was made. These technique are speicfic to the architecture of the model and its implementation details.
* Explainability involves external methods to understand the model performance. These are usually agnostic to the specifics of the trained model.

This is still an emerging area which will grow in importance in coming years to build trust in the use of machine learning for research and operational purposes. The tools are a lot less mature and so difficult to incorporate into a brief tutorial like this. More information about this tppic is available through the links below.

Further information
- [ Interpretable ML by CHristopher Monar - ebook](https://christophm.github.io/interpretable-ml-book/)
- [The difference between interpretability and explainability](https://milvus.io/ai-quick-reference/what-is-the-difference-between-interpretability-and-explainability)

## 11. Model Storage
We have demonstrated training a very small ML model. For larger models such global weather and climate models, the training costs are much higher. Thus we expect to train a model once and then use it many times subsequently. Thus we need to save the parameters and architecture of the model, ater training, so it can subsequently be loaded and reused for predictions. Different models have different elements that need to be stored. Some examples include:

- Decision tree - The nodes, the connections between nodes and the decision criteria for each node.
- Neural network - The number and type of nodes in each layer, connections between nodes (if not fully connected or including skip connections) and the weights for each of the nodes.

A simple way to save the model in scikit-learn is to use the python pickle function, which can create a serialized version of (almost) any in memory object which can then be saved and loaded to or from a storage device (e.g. a hard disk). Other ML libraries have other formats for storing model definitons and there are some third party formats that work with many different libraries.

Further reading
* [scikit learn model storage](https://scikit-learn.org/stable/model_persistence.html)
* [ONNX docs](https://onnx.ai/onnx/)

### Loading and storing with ML Flow
One additional feature of usingn ML flow for tracking our experiments, is that the trained model is saved as a part of the 

### Unsupervised learning - clustering

So far in this tutorial, we have been doing *supervised learning*, which is where we show the algortihms examples which match inputs and desried outputs. Another form of learning is where we don't have the "right answer" and instead the algorithm learns patterns in the data. In this dataset, we could find patterns in the climate means to come up with a different climate zone classification scheme that is learned from the data

In [ ]:
km_clusterer = sklearn.cluster.KMeans(n_clusters=5)
km_clusterer

In [ ]:
km_clusterer.fit(X_train)

In [ ]:
km_clusterer.predict(X_train)a

In [ ]:
#todo - plots of data on a map

## Exercises

Once you have worked through the tutorial above, you can test your knowledge by adapting the code in the tutorial to experiment with different options for training the model and comparing results.

### Historic and future comparison
So far we have put all the data from historic and future scenarios together into one big dataset. The mapping from climate variables to climate zones should be the same in the future, even as the climate zones of particular places changes. We can test that by using historic data as our training set and then use future scenarios as our test data.  

In [ ]:
# insert  code here

### Divide train/test by geographic region
Doing our train/test split randomly, we are likely to have lots of correlations between examples in the train/val/test sets. One way we could reduce this is by spliting by geographic region. As the zones are highly corrlated with latitude, we couldn't use latitude for splitting. Instead we could divide by longitude. We would need to ensure all zones are represented in train, val and test sets.

In [ ]:
# insert  code here

### Addressing class imbalance
We have seen that the number of memebers of different classes 

In [ ]:
# insert  code here

In [ ]:
# train on historic data and predict on future data

In [ ]:
# try to divide train/test by geographic region. Use longitude 

In [ ]:
# try to balance classes in train set and compare results

### Next steps or potential follow on material

Additional excercises in this tutorial material includes:
- Training a neural network using pytorch
- Training a CNN autoencoder on CMIP6 data
- Training a RNN on Jena weather dataset for time series prediction. 


###  Exmaples of Use

### Data statement
###     References
